# LC 901 — Online Stock Span
**Day-71 | Monotonic Stack | Medium**

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> Every price "absorbs" all consecutive
previous prices that are smaller than or equal to it.
A monotonic <em>decreasing</em> stack of (price, span) lets us
collapse that history in O(1) amortised time — each element
is pushed and popped at most once across all calls.
</div>

## Official Problem Statement

Design a class `StockSpanner` that collects daily price quotes for a
stock and returns the **span** of that stock's price for the current
day.

The span of the stock's price today is defined as the maximum number
of consecutive days (starting from today and going backwards)
for which the stock price was **less than or equal to** today's price.

**API:**
```
StockSpanner()
int next(int price)
```

**Constraints:**
- `1 <= price <= 10^5`
- At most `10^4` calls to `next`.

## What This Is Actually Asking

For each incoming price, count how many days in a row (including
today, going backwards) had a price ≤ today's price.

- Day 1: price=100 → span=1 (just today)
- Day 2: price=80  → span=1 (100 > 80, stop)
- Day 3: price=60  → span=1 (80 > 60, stop)
- Day 4: price=70  → span=2 (60 ≤ 70, then 80 > 70, stop)
- Day 5: price=60  → span=1 (70 > 60, stop)
- Day 6: price=75  → span=4 (60,70,60 all ≤ 75, then 80 > 75)
- Day 7: price=85  → span=6 (75,60,70,60,80 all ≤ 85, then 100>85)

The naive solution scans backwards each time — O(n) per call.
The trick: **store accumulated spans on the stack** so we can
skip whole chunks at once.

## Walk Through an Example by Hand

Prices arrive: 100, 80, 60, 70, 60, 75, 85

```
call next(100): span=1, stack=[(100,1)]          → return 1
call next(80) : span=1, 100>80 stop
                stack=[(100,1),(80,1)]            → return 1
call next(60) : span=1, 80>60 stop
                stack=[(100,1),(80,1),(60,1)]     → return 1
call next(70) : span=1
  pop (60,1)  : 60<=70, span=1+1=2
  80>70 stop
                stack=[(100,1),(80,1),(70,2)]     → return 2
call next(60) : span=1, 70>60 stop
                stack=[...,(70,2),(60,1)]         → return 1
call next(75) : span=1
  pop (60,1)  : 60<=75, span=2
  pop (70,2)  : 70<=75, span=4
  80>75 stop
                stack=[(100,1),(80,1),(75,4)]     → return 4
call next(85) : span=1
  pop (75,4)  : 75<=85, span=5
  pop (80,1)  : 80<=85, span=6
  100>85 stop
                stack=[(100,1),(85,6)]            → return 6
```

## The Picture

Bar chart of prices (each `#` = 10 units):

```
100 | ##########
 85 |           ########## <-- day 7, span=6
 80 |  ########
 75 |           ########
 70 |    ########
 60 |     ######     ######
     D1   D2   D3   D4   D5   D6   D7
```

Stack state after each call (top is right):

```
after next(100): [(100,1)]
after next(80) : [(100,1), (80,1)]
after next(60) : [(100,1), (80,1), (60,1)]
after next(70) : [(100,1), (80,1), (70,2)]  <-- (60,1) absorbed
after next(60) : [(100,1), (80,1), (70,2), (60,1)]
after next(75) : [(100,1), (80,1), (75,4)]  <-- (60,1),(70,2) gone
after next(85) : [(100,1), (85,6)]          <-- (80,1),(75,4) gone
```

Key observation: stack is always **strictly decreasing** by price.
Each entry's span = how many consecutive days it "represents".

## When To Use This Pattern

Use a **monotonic stack** when:
- You need the nearest greater/smaller element (left or right)
- You need to count consecutive elements satisfying a condition
- The answer for the current element depends on "how far back"
  you can go before hitting a blocker

**Signals in the problem:**
- "span", "consecutive days", "streak"
- "next greater", "previous smaller"
- Online processing (one element at a time)

**Variant clues:**
- Stack of values → basic next greater
- Stack of (value, count/span) → aggregate history
- Stack of indices → when you need position info too

## The Approach

**Data structure:** Stack of `(price, span)` tuples.

**Algorithm for `next(price)`:**
1. Start `span = 1`
2. While stack is not empty AND `stack[-1][0] <= price`:
   - Pop `(p, s)` from stack
   - Add `s` to `span`  ← absorb that day's whole streak
3. Push `(price, span)` onto stack
4. Return `span`

**Why it works:**
- Every element is pushed once and popped at most once → O(1) amortised
- The span stored on the stack is the *pre-accumulated* count,
  so we jump over entire flat/downward stretches in one step.

**Complexity:** O(1) amortised per call, O(n) space.

In [1]:
from typing import List

In [4]:
# ── Test harness (op-replay style for design problems) ──────────────

def test_harness_901(SpannerClass):
    """
    Replays a sequence of operations on StockSpanner and compares
    results to expected output. Prints PASSED / FAILED per case.
    """
    test_cases = [
        {
            "ops":    ["next","next","next","next",
                       "next","next","next"],
            "args":   [100, 80, 60, 70, 60, 75, 85],
            "expect": [1, 1, 1, 2, 1, 4, 6],
            "label":  "Example 1 (LeetCode)",
        },
        {
            "ops":    ["next","next","next"],
            "args":   [30, 30, 30],
            "expect": [1, 2, 3],
            "label":  "All equal prices",
        },
        {
            "ops":    ["next","next","next"],
            "args":   [10, 20, 30],
            "expect": [1, 2, 3],
            "label":  "Strictly increasing",
        },
        {
            "ops":    ["next","next","next"],
            "args":   [30, 20, 10],
            "expect": [1, 1, 1],
            "label":  "Strictly decreasing",
        },
    ]

    passed = 0
    for tc in test_cases:
        spanner = StockSpanner()
        results = [spanner.next(p) for p in tc["args"]]
        ok = results == tc["expect"]
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(f"  [{status}] {tc['label']}")
        if not ok:
            print(f"    got:      {results}")
            print(f"    expected: {tc['expect']}")

    total = len(test_cases)
    print(f"\n  Summary: {passed}/{total} passed")


print("Test harness defined.")

Test harness defined.


In [6]:
class StockSpanner:
    """
    LC 901 — Online Stock Span

    Approach: Monotonic decreasing stack of (price, span) tuples.

    next(price):
      - span starts at 1 (today counts)
      - Pop all stack entries whose price <= current price,
        accumulating their spans.
      - Push (price, span) onto the stack.
      - Return span.

    Amortised O(1) per call — each element pushed/popped once.
    Space: O(n) for the stack.

    Args:
        None (constructor)

    Returns:
        int: span of the stock price for the current day.
    """

    def __init__(self):
        # Stack stores (price, accumulated_span) tuples.
        # Maintained in strictly decreasing order of price.
        self.stack = []

    def next(self, price: int) -> int:
        """
        Args:
            price (int): today's stock price (1 <= price <= 10^5)

        Returns:
            int: number of consecutive days (including today,
                 going backwards) with price <= today's price.
        """
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack[-1][1]
            self.stack.pop()
        self.stack.append([price, span])
        return span


# --- Debug prints (expected in comments) ---
s = StockSpanner()
print(s.next(100))  # 1
print(s.next(80))   # 1
print(s.next(60))   # 1
print(s.next(70))   # 2
print(s.next(60))   # 1
print(s.next(75))   # 4
print(s.next(85))   # 6

print("---")
s2 = StockSpanner()
print(s2.next(100))  # 1
print(s2.next(100))  # 2  (equal counts!)
print(s2.next(100))  # 3

test_harness_901(StockSpanner)

1
1
1
2
1
4
6
---
1
2
3
  [PASSED] Example 1 (LeetCode)
  [PASSED] All equal prices
  [PASSED] Strictly increasing
  [PASSED] Strictly decreasing

  Summary: 4/4 passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness_901()

## Complexity

| | Time | Space |
|---|---|---|
| `__init__` | O(1) | O(1) |
| `next` (amortised) | **O(1)** | — |
| `next` (worst single call) | O(n) | — |
| Overall after n calls | O(n) total | O(n) |

**Why amortised O(1)?**
Each price is pushed exactly once and popped at most once.
Across all `n` calls: ≤ n pushes + ≤ n pops = O(n) total work,
so O(1) per call on average.

**Space:** The stack holds at most one entry per `next` call
that has not yet been dominated → O(n) worst case (strictly
decreasing prices).

## Real World Connection

**Financial analytics dashboards** use exactly this pattern.

When a trading platform displays "X-day high" indicators in
real time, it needs the span for each incoming tick without
rescanning history. The monotonic stack is embedded inside
streaming aggregation engines (e.g., Apache Flink window
operators, AWS Kinesis Analytics sliding windows).

The same idea appears in:
- **Candlestick charting:** how many bars back is this the
  highest close?
- **Network monitoring:** how many consecutive seconds has
  this metric stayed below the threshold?
- **Sensor data:** streak detection in IoT pipelines.

The key engineering insight is identical: accumulate history
lazily in a stack so each new event costs O(1) amortised,
enabling high-throughput real-time processing.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra